<a href="https://colab.research.google.com/github/chaitanyatalakeri27-png/Data_Science_Lab/blob/main/Exp9_Triplet_Loss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Flatten, Concatenate

# Parameters
embedding_dim = 16
input_shape = (32, 32)

# Embedding Network
def create_embedding_network():

    model_input = Input(shape=input_shape)

    x = Flatten()(model_input)

    x = Dense(64, activation='relu')(x)
    x = Dense(32, activation='relu')(x)
    output = Dense(embedding_dim)(x)

    model = Model(model_input, output)

    return model

# Create Embedding Model
embedding_network = create_embedding_network()

# Inputs
anchor_input = Input(shape=input_shape, name='anchor_input')
positive_input = Input(shape=input_shape, name='positive_input')
negative_input = Input(shape=input_shape, name='negative_input')

# Generate Embeddings
anchor_embedding = embedding_network(anchor_input)
positive_embedding = embedding_network(positive_input)
negative_embedding = embedding_network(negative_input)

# THE FIX: Concatenate the 3 outputs into a SINGLE tensor so Keras doesn't freak out
merged_output = Concatenate(axis=1)([
    anchor_embedding,
    positive_embedding,
    negative_embedding
])

# Triplet Model
triplet_model = Model(
    inputs=[anchor_input, positive_input, negative_input],
    outputs=merged_output # Outputting the single merged tensor
)

# Triplet Loss Function
def triplet_loss(y_true, y_pred):

    # THE FIX: Split the merged tensor back into 3 pieces inside the loss function
    anchor = y_pred[:, 0:embedding_dim]
    positive = y_pred[:, embedding_dim:2*embedding_dim]
    negative = y_pred[:, 2*embedding_dim:3*embedding_dim]

    positive_distance = tf.reduce_sum(tf.square(anchor - positive), axis=1)
    negative_distance = tf.reduce_sum(tf.square(anchor - negative), axis=1)

    margin = 0.5

    loss = tf.maximum(positive_distance - negative_distance + margin, 0.0)

    return tf.reduce_mean(loss)

# Compile Model
triplet_model.compile(
    optimizer='rmsprop',
    loss=triplet_loss
)

# Generate Dummy Data
num_samples = 200

anchor_data = np.random.normal(0, 1, (num_samples, 32, 32))
positive_data = np.random.normal(0, 1, (num_samples, 32, 32))
negative_data = np.random.normal(0, 1, (num_samples, 32, 32))

dummy_labels = np.zeros((num_samples,))

# Train Model
history = triplet_model.fit(
    [
        anchor_data,
        positive_data,
        negative_data
    ],
    dummy_labels,
    epochs=3,
    batch_size=64
)

print("\nOptimized Triplet Loss Model Trained Successfully!")

Epoch 1/3
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 3.0851  
Epoch 2/3
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1101 
Epoch 3/3
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0029 

Optimized Triplet Loss Model Trained Successfully!
